# Word Embeddings

**Notebook 2 of 3.** In notebook 01 every word was a column of its own, so `sea` and `ocean` were as different as `sea` and `bee`: the columns know nothing about meaning.

A **word embedding** gives every word a short vector, learned from the text so that words used in similar places get similar vectors. The idea goes back to the linguist J. R. Firth (1957): *"You shall know a word by the company it keeps."*

Here you train them with a small neural network, for 500 words, so every step stays visible; notebook 03 does the same for a whole vocabulary with **word2vec**. Both follow these steps:

<img src="../assets/embedding_pipeline.png" alt="Five steps: a sentence split into words, its training pairs, a network with a one-hot input and two hidden neurons, the learned vectors, and a plot in which the whale words lie apart from the horse words." width="1000">

## Setup

In [ ]:
# standard library
import os
import re
from collections import Counter
from pathlib import Path

# TensorFlow reports what it finds on this machine while it loads. This setting hides most
# of it and has to run before Keras imports TensorFlow, which is why it sits here.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# data stack
import numpy as np
import pandas as pd

# visualization stack
import matplotlib.pyplot as plt

# deep learning stack
# The last few of TensorFlow's notes go straight to the error stream, below the level the
# setting above reaches. Point that stream at nothing while Keras imports it, then put it
# back, so the notebook starts clean.
with open(os.devnull, "w") as quiet:
    stderr = os.dup(2)
    os.dup2(quiet.fileno(), 2)
    try:
        import keras
    finally:
        os.dup2(stderr, 2)
        os.close(stderr)

# ml stack
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. The Words to Embed

A word's vector is learned from the words beside it, and two numbers per word can only show a structure strong enough to survive being flattened onto a plane. The poems of notebook 01 offer none — they jump from subject to subject — so this notebook uses four handbooks instead, two about each of two animals that share no surroundings:

| Animal | Books |
|---|---|
| Whales | a NOAA field guide to the whales, dolphins and porpoises of the western North Atlantic, and a primer on the California gray whale |
| Horses | the horse chapter of a book on farm animals, and the US Department of Agriculture's report on the diseases of the horse |

`data/animals.csv` holds them one paragraph per row, the two sides cut to the same number of words so that neither animal dominates; `data/build_animals.py` documents how.

In [ ]:
animals = pd.read_csv(Path("../data") / "animals.csv")
print(animals["subject"].value_counts().to_string())
animals.head(3)

Tokenize and drop the stop words: they stand next to almost every word and so say nothing about any one of them ([`ENGLISH_STOP_WORDS`](https://scikit-learn.org/stable/modules/feature_extraction.html#stop-words)).

The pattern is `\b[a-z][a-z]+\b`, letters only, rather than notebook 01's `\b\w\w+\b`: a field guide is full of measurements, and `\w` matches digits, so the looser pattern would learn a vector for `12`.

In [ ]:
def tokenize(text):
    """Return the lowercase word tokens of a text without the English stop words."""
    tokens = []
    for token in re.findall(r"\b[a-z][a-z]+\b", text.lower()):
        if token not in ENGLISH_STOP_WORDS:
            tokens.append(token)
    return tokens


sample = "Blue whales may reach 100 feet in length."
print("All tokens:", re.findall(r"\b\w\w+\b", sample.lower()))
print("Kept:     ", tokenize(sample))

In [ ]:
# Tokenize every paragraph: one list of tokens per paragraph
documents = animals["text"].map(tokenize)
print(f"{len(documents):,} documents, {documents.str.len().sum():,} tokens")
print("First 10 tokens of the first paragraph:", documents.iloc[0][:10])

The four books hold about 8,000 different words. This notebook keeps the **500 most frequent**, for one reason: a vector is learned from the pairs a word appears in, so a word seen three times lands anywhere. The 500th still occurs 20 times.

Every word of the vocabulary is both a **target**, whose vector is learned, and a **context** word to be predicted, and a network computes with numbers, so each gets one: its position in the vocabulary.

In [ ]:
# How often each word occurs in all paragraphs together
counts = Counter()
for tokens in documents:
    counts.update(tokens)

print(f"{len(counts):,} different words")
counts.most_common(6)

In [ ]:
# The 500 most frequent words, and a number for every one of them
vocabulary = []
for word, count in counts.most_common(500):
    vocabulary.append(word)
word_index = {}
for i, word in enumerate(vocabulary):
    word_index[word] = i

print(f"{len(vocabulary)} words, from {vocabulary[:3]} to {vocabulary[-3:]}")
print(f"The rarest of them occurs {counts[vocabulary[-1]]} times. Number of whale:", word_index["whale"])

## 2. Training Pairs: the Skip-Gram Approach

The task is simple: given a word, the **target**, predict the words around it, its **context words**. A **window** of size 5 takes up to five words each side, every pair in it is one training pair, and the window moves on. This is **skip-gram** (Mikolov et al., 2013); word2vec also offers **CBOW**, which predicts the target from its context words at once, and notebook 03 compares them.

<img src="../assets/skip_gram_window.png" alt="The sentence blue whales usually occur singly pairs, with whales as the target and a window over the whole sentence, giving five target-context pairs." width="760">

The window does not stop at a sentence end: in the paragraph this sentence closes, four more words reach `whales` from the sentence before. The cell below carries two of them along to show that.

A word has different neighbors in different paragraphs, so the prediction can never be perfect. To do well on average, the network must give words with similar neighbors similar vectors.

The cell below cuts that window, with those two earlier words added. A list slice `line[a:b]` returns the words from position `a` up to, but not including, position `b`, and `max(0, ...)` keeps the start from running off the front. Every target needs the same two slices, so they go in a function.

In [ ]:
line = ["surface", "feet", "blue", "whales", "usually", "occur", "singly", "pairs"]


def window_of(tokens, position, window):
    """Return up to `window` words before and after `position`, as one list."""
    before = tokens[max(0, position - window):position]
    after = tokens[position + 1:position + window + 1]
    return before + after


print("Target:", line[3], "| window:", window_of(line, position=3, window=5))

Two more functions finish the job, each doing one thing:

- `pairs_in` first drops the words of a document that are not in the vocabulary, then takes the window around each word that is left. **The order matters.** Discarding them afterwards would let a word that gets no vector still take up a place in the window, so the neighbors kept would be words that stood further apart. Dropping first is what word2vec does too. [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate) gives each word with its position.
- `skip_gram_pairs` runs that over every paragraph and collects the rows into a DataFrame.

In [ ]:
def pairs_in(tokens, window):
    """Return the (target, context) pairs of one document, over its vocabulary words."""
    kept = [token for token in tokens if token in word_index]  # drop what has no vector first
    pairs = []
    for position, target in enumerate(kept):
        pairs += [(target, context) for context in window_of(kept, position, window)]
    return pairs


def skip_gram_pairs(documents, window):
    """Return a DataFrame with one row per (target, context) pair of every document."""
    rows = [pair for tokens in documents for pair in pairs_in(tokens, window)]
    return pd.DataFrame(rows, columns=["target", "context"])


# The same sentence as above: every word of it that is in the vocabulary takes its turn as target
print("Dropped:", [word for word in line if word not in word_index])
skip_gram_pairs([line], window=5)

`singly` and `pairs` are not among the 500 words, so `pairs_in` drops them before it cuts any window, and `whales` keeps five of its seven pairs: `surface`, `feet`, `blue`, `usually` and `occur` — its true nearest neighbors among the words that have a vector.

In [ ]:
pairs = skip_gram_pairs(documents, window=5)
print(f"{len(pairs):,} pairs")
print("Pairs per target word:", pairs["target"].value_counts().head(4).to_dict())
pairs.head(8)

## 3. Encode the Pairs

The input of the network is the **one-hot vector** of the target: 500 zeros with a single 1 at the position of the word. `np.eye(n)` is the $n \times n$ identity matrix, whose row $i$ is the one-hot vector of word $i$, so indexing it with a list of word numbers encodes the whole list at once ([`np.eye`](https://numpy.org/doc/stable/reference/generated/numpy.eye.html)). `dtype="float32"` halves the cost, and is what Keras converts to anyway.

The output is the number of the context word, as for any multiclass classification with integer labels.

In [ ]:
n_words = len(vocabulary)
# The number of the target word of every pair
target_numbers = pairs["target"].map(word_index)
target_numbers.head(3)

In [ ]:
# Row i of the identity matrix is the one-hot vector of word number i
X = np.eye(n_words, dtype="float32")[target_numbers]
print(f"X: {X.shape}, {X.nbytes / 1e6:.0f} MB")
print(f"Pair 0, target {pairs['target'].iloc[0]!r}:", X[0].astype(int))

In [ ]:
# The label of every pair: the number of its context word
y = pairs["context"].map(word_index).to_numpy()
print("y:", y.shape)
print(f"Pair 0, context {pairs['context'].iloc[0]!r}: number {y[0]}")

That matrix is almost entirely zeros, and it is the reason this notebook stops at 500 words: a vocabulary of 1,000 would already need 1.2 GB, most of it zeros. Word2vec, in notebook 03, never builds it.

## 4. The Embedding Network

Three layers:

- **input** — the one-hot vector of the target, 500 numbers;
- **hidden** — 2 neurons, whose two outputs are the embedding;
- **output** — 500 neurons with a softmax, the predicted probability of each context word.

<img src="../assets/embedding_network.png" alt="500 one-hot inputs, two hidden neurons without activation, 500 softmax outputs, with the two weights of whale highlighted." width="720">

Each input word reaches the 2 hidden neurons by its own 2 weights (blue for `whale`). For a one-hot vector every input is 0 but one, so the hidden layer outputs exactly those 2 weights: the embedding of `whale`, its position in a plot. Training adjusts them until the output predicts `whale`'s neighbors as well as it can, and words with similar neighbors need similar numbers for that.

<details>
<summary>Show details: the weight matrix and the loss</summary>

The 2 weights of each word form a $500 \times 2$ matrix $W$. Two choices make the hidden output exactly one row of it: **no activation** (`activation=None`), so the output is $x W$, and **no bias** (`use_bias=False`). For the one-hot $x$ of word $i$, $x W$ zeroes every row but $i$.

The output layer is an ordinary multiclass classifier, so the loss is `sparse_categorical_crossentropy`. Two dimensions are far too few for real embeddings, but they let you plot every word directly.

</details>

Training takes a few seconds.

In [ ]:
# clear_session and set_random_seed together make the training below reproducible
keras.utils.clear_session()
keras.utils.set_random_seed(42)

embedding_size = 2
model = keras.Sequential(
    [
        keras.Input(shape=(n_words,)),
        keras.layers.Dense(units=embedding_size, activation=None, use_bias=False, name="embedding"),
        keras.layers.Dense(units=n_words, activation="softmax", name="context"),
    ]
)

The summary below should show no bias for the embedding layer, and $500 \times 2$ weights in it: one row per word.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="sparse_categorical_crossentropy")
model.summary()

In [ ]:
history = model.fit(X, y, epochs=10, batch_size=256, verbose=0)
losses = pd.Series(history.history["loss"])
losses.plot(xlabel="Epoch", ylabel="Training loss", title="Embedding network", figsize=(7, 3.5))
plt.show()
print(f"Loss after the first epoch: {losses.iloc[0]:.3f}, after the last: {losses.iloc[-1]:.3f}")

In [ ]:
# Two losses to compare with: a uniform guess, and a guess by overall frequency
print(f"Loss of a uniform guess over {n_words} words: ln({n_words}) = {np.log(n_words):.3f}")
frequencies = pairs["context"].value_counts(normalize=True)
print(f"Loss of predicting every context word with its overall frequency: {-(frequencies * np.log(frequencies)).sum():.3f}")

**Result.** The loss starts at 5.652, below $\ln(500) = 6.215$, the loss of a network that gives all 500 words the same probability, and falls to 5.529 within about three epochs; the rest of the curve is flat. It can never reach 0, because `whale` appears next to hundreds of different words.

The number to compare against is **5.880**, the loss of predicting every context word with its overall frequency, whatever the target: that needs no embeddings at all, since some words are simply likely neighbors of anything. Everything below it is what the 2 numbers per word bought. The margin is thin, and it has to be: two numbers have to serve every paragraph the word appears in.

## 5. Read the Embeddings from the Weights

Predicting context words was only a means to an end. What you keep are the hidden layer's weights: 2 numbers for each of the 500 words, which are its coordinates in a plot.

In [ ]:
weights = model.get_layer("embedding").get_weights()[0]
# Keras trains in float32; as float64 the tiny distances of section 6 keep their digits
embeddings = pd.DataFrame(weights, index=vocabulary, columns=["x", "y"]).astype(float)
print("Embedding matrix:", embeddings.shape)
embeddings.head(4).round(2)

Those rows are what the details box above promised: the hidden layer's output for a one-hot input is exactly that word's row of the matrix.

In [ ]:
whale = np.eye(n_words, dtype="float32")[[word_index["whale"]]]  # the one-hot vector of whale, as a batch of one
hidden_output = np.asarray(model.get_layer("embedding")(whale))[0]
print("Hidden output for whale:", hidden_output.round(3))
print("Row of whale:          ", embeddings.loc["whale"].to_numpy().round(3))

The two weights are the word's coordinates, so the 500 words can be plotted as they are. The network was never told which book a word came from, so count that separately and color the points by it. Labels go on the most frequent words, but only where one does not land on another, since 500 labels would cover each other. The plot is centered on $(0, 0)$, because section 6 compares directions seen from there.

In [ ]:
# Which of the two animals a word is written about more often
by_subject = {}
for subject, group in animals.groupby("subject"):
    by_subject[subject] = Counter(token for text in group["text"] for token in tokenize(text))

written_about = pd.Series(
    {word: "whales" if by_subject["whales"][word] >= by_subject["horses"][word] else "horses" for word in vocabulary}
)
written_about.value_counts()

In [ ]:
def label_fits(word, placed):
    """Return True if the label of `word` would not land on the label of any word in `placed`."""
    for other in placed:
        apart_x, apart_y = (embeddings.loc[word] - embeddings.loc[other]).abs()
        if apart_x < 0.85 and apart_y < 0.17:  # roughly the width and height of a label here
            return False
    return True


# Work down the vocabulary by frequency, labeling every word whose label still has room
labeled = []
for word in vocabulary:
    if label_fits(word, labeled):
        labeled.append(word)
print(f"{len(labeled)} of the 500 words are labeled")

fig, ax = plt.subplots(figsize=(9, 9))
for subject, color in [("whales", "tab:blue"), ("horses", "tab:orange")]:
    words = written_about.index[written_about == subject]
    ax.scatter(embeddings.loc[words, "x"], embeddings.loc[words, "y"], s=18, color=color, alpha=0.7, label=subject)
for word in labeled:
    ax.annotate(word, embeddings.loc[word], xytext=(4, 3), textcoords="offset points", fontsize=9)

# Center on (0, 0): the cosine similarity compares directions seen from the origin
limit = 1.1 * embeddings.abs().max().max()
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.legend(title="Written about more often in the books on")
ax.set(xlim=(-limit, limit), ylim=(-limit, limit),
       xlabel="First weight", ylabel="Second weight",
       title="The 500 word vectors, labeled where the labels fit")
plt.show()

**Result.** Two clouds, and the colors were added afterwards: the network never saw which book a word came from. It saw only which words stand near which, and that alone split the plane down the vertical axis, most of the 211 words of the horse books to the left of it and most of the 289 of the whale books to the right.

Read the labels and the two subjects are unmistakable. Left: `oats`, `fed`, `feed`, `horse`, `horses`, `disease`, `inflammation`, `blood`, `discharge`, `bones`, `chest`. Right: `whales`, `dolphins`, `killer`, `whaling`, `migration`, `fin`, `dorsal`, `teeth`, `north`, `florida`.

The words sitting on the axis are the ones both sets of books use: `animal`, `large`, `long`, `occur`. A word's place is decided by the company it keeps, and those keep both companies.

## 6. Similar Words

The **cosine similarity** of two vectors $a$ and $b$ measures how much they point in the same direction:

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

It is 1 for the same direction, 0 for perpendicular vectors and $-1$ for opposite ones.

The **cosine distance** is $1 - \cos(a, b)$: 0 for the same direction, 2 for opposite. Rankings use it, and here you need it: 500 words share one plane, the nearest neighbors sit well under a degree apart, and 0.99999 and 0.99997 both print as 1.00 while their distances, 0.00001 and 0.00003, stay apart. They are small enough that the cell prints them in scientific notation.

In [ ]:
def cosine_similarity(a, b):
    """Return the cosine similarity of two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def cosine_distance(a, b):
    """Return the cosine distance of two vectors: 0 for the same direction, 2 for opposite ones."""
    return 1 - cosine_similarity(a, b)


# The same relation said three ways, for one pair of words
whale_vector, dolphins_vector = embeddings.loc["whale"], embeddings.loc["dolphins"]
similarity = cosine_similarity(whale_vector, dolphins_vector)
distance = cosine_distance(whale_vector, dolphins_vector)
print(f"cosine similarity of whale and dolphins: {similarity:.6f}")
print(f"cosine distance:                         {distance:.6f}")
print(f"angle between the two vectors:           {np.degrees(np.arccos(similarity)):.1f} degrees")

**Your turn:** complete `most_similar`, which ranks all other words of the vocabulary by their cosine distance to a given word, closest first.

In [ ]:
def most_similar(word, vectors, top_n=5):
    """Return the top_n words of the vectors DataFrame closest to word, with their cosine distance."""
    target = vectors.loc[word]
    distances = pd.Series(dtype=float)
    for other in vectors.index:
        distances[other] = cosine_distance(vectors.loc[other], target)
    return distances.drop(word).nsmallest(top_n)


for word in ["whale", "fin", "horse", "feed"]:
    closest = most_similar(word, embeddings)
    print(f"{word:6s}", ", ".join(f"{other} {distance:.1e}" for other, distance in closest.items()))

**Result.** The distances run from 3.3e-11 to 6.0e-04 and no two of them are alike, which is exactly what the similarity could not deliver: all twenty of these words have a cosine of 1.00 to two decimal places. The same information, written so it can be read.

Not one of the twenty neighbors is a synonym, and every one comes from the right book: `killer`, `saddleback` and `bowhead` are whales, `falcate` and `tall` describe a dorsal fin, `rectum` and `physic` are where the veterinary report goes.

`figure` is the instructive one, third for `whale` and no part of a whale at all. It is there because 161 of the guide's paragraphs are captions beginning *Figure 4. A fin whale...*, so it stands beside `whale` 164 times and beside `horse` twice. That is all an embedding measures: words **used** alike come out alike, which is not words that **mean** alike.

`most_similar` ranks the whole vocabulary against one word. The cell below instead measures a few words against **two** of them, `whale` and `horse`, and so reads the plot as numbers: how far each word stands from each pole.

In [ ]:
def angle_to(word, pole):
    """Return the angle in degrees between the vectors of two words."""
    cosine = cosine_similarity(embeddings.loc[word], embeddings.loc[pole])
    return np.degrees(np.arccos(np.clip(cosine, -1, 1)))


poles = pd.DataFrame(
    {pole: [angle_to(word, pole) for word in ["dolphins", "dorsal", "fin", "feed", "disease", "blood", "animal", "body"]]
     for pole in ["whale", "horse"]},
    index=["dolphins", "dorsal", "fin", "feed", "disease", "blood", "animal", "body"],
)
poles.round(0).astype(int)

**Result.** The table reads the plot as numbers. `dolphins` stands 6 degrees from `whale` and 176 from `horse`; `fin` and `dorsal` sit near the same pole and far from the other. `feed`, `disease` and `blood` do the same the other way round. `body` is the word in the middle, 88 degrees from `whale` and 102 from `horse` — near a right angle from both, which is what a word used by both sets of books looks like with only two dimensions to place it in.

## Summary

- An embedding is a short vector per word, learned so that words used in similar places get similar vectors.
- Skip-gram builds (target, context) pairs with a sliding window. For a one-hot input the hidden layer outputs that word's weights, and those weights **are** its embedding.
- The cosine similarity compares two directions; the cosine distance is $1 - \cos$. In two dimensions the similarity saturates, so rank by the distance. Both find words used alike, not words that mean alike.
- Two dimensions and 500 words keep every step visible, but the ranking is coarse and the one-hot matrix already costs half a gigabyte.

Next, meet vectors trained on six billion words in [03 - Word2vec](03_word2vec.ipynb).

## Sources

- The four handbooks are Project Gutenberg eBooks [#33527](https://www.gutenberg.org/ebooks/33527), [#58695](https://www.gutenberg.org/ebooks/58695), [#34175](https://www.gutenberg.org/ebooks/34175) and [#23403](https://www.gutenberg.org/ebooks/23403), public domain in the USA; `data/build_animals.py` shows how `animals.csv` was cut from them.
- The network follows the embeddings notebook of the neuefische Intro to NLP repository.
- The move this notebook opens with, from the bag of words to dense vectors, is the one chapter 1 of *Hands-On Large Language Models* (Alammar and Grootendorst, O'Reilly, 2024) makes, in that order and for that reason.

The repository [README](README.md) lists every source and licence.